# 2일차 · 80%가 되는 때를 미리 알기 — 머신러닝
**충북대학교 · Python 활용 머신러닝 및 딥러닝 특강** 

어제는 용량 곡선을 **끝까지 보고** 80% 지점을 찾았습니다. 오늘은 **앞부분만 보고**, 또는 **다른 셀만 보고** 그 지점을 **미리** 맞힙니다.
위에서부터 한 셀씩 실행합니다. 타이핑은 없고 `# ←` 자리의 **숫자와 이름만** 바꿉니다.

| 막히면 | 이렇게 |
|---|---|
| 빨간 오류 | 오류 아래 **「오류 설명」** 버튼 → 그래도 모르면 오류 글을 복사해 Zoom 채팅에 |
| ✅ 표시 | 결과 숫자를 **Zoom 채팅**에 올립니다 |

먼저 **파일 → Drive에 사본 저장**.

## 블록 1 · 학습이란 무엇인가

### 0. 어제 데이터 다시 읽기
이 셀이 하는 일 — 어제 5번 셀과 같습니다. 표를 읽고 B0005만 `b5`로 골라 둡니다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

BASE = "https://raw.githubusercontent.com/aicec/battery-ml-data/main/"
summary = pd.read_csv(BASE + "cycles_summary.csv")
b5 = summary[summary.cell_id == "B0005"]

print(len(summary), "행 ·", summary.cell_id.unique().tolist())

### 1. 내가 끈 직선 — 브라우저 활동
브라우저에서 **직선 끌기 페이지**를 엽니다 → https://aicec.github.io/battery-ml-data/line.html
B0005의 **앞 60사이클**만 보입니다. 기울기 · 절편 슬라이더를 움직여 점들 사이로 직선을 맞추고, **오차(MAE)** 를 가장 작게 만들어 보세요.

✅ **체크포인트 ①** — 페이지에 보이는 **내 오차**(예: `0.0180`)를 채팅에 올려 주세요.

이 셀이 하는 일 — 페이지에서 찾은 기울기 `a` 와 절편 `b` 를 적으면, 같은 오차를 코드로 다시 계산하고 그 직선이 **1.6 Ah(80%)에 닿는 사이클**을 구합니다.
`예측 용량 = a × 사이클 + b` · **오차(MAE) = |실제 − 예측| 의 평균**

In [ ]:
a = -0.0020   # ← 페이지의 기울기
b = 1.860     # ← 페이지의 절편

train = b5[b5.cycle <= 60]                    # 앞 60사이클
pred  = a * train.cycle + b                   # 내 직선이 말하는 용량
mae   = (train.capacity_Ah - pred).abs().mean()

print(f"내 직선의 오차(MAE): {mae:.4f} Ah")
print(f"내 직선이 1.6 Ah에 닿는 사이클: {(1.6 - b) / a:.0f}")

**Zoom 투표(✅ ②)** — 기출문제는 다 맞히는데 모의고사는 못 보는 학생은 누구일까요? (학생 A · B · C)

| | 기출 점수 | 모의고사 점수 |
|---|---|---|
| 학생 A | 100 | 60 |
| 학생 B | 85 | 83 |
| 학생 C | 60 | 58 |

머신러닝 말로 — 기출 = **학습 데이터(train)**, 모의고사 = **검증 데이터(test)**, 학생 A = **과적합**(답만 외움).

## 블록 2 · 선형회귀 — 컴퓨터가 직선을 찾는다

### 2. 기출과 모의고사로 나누기
이 셀이 하는 일 — B0005를 **앞 60사이클(학습용)** 과 **61회차 이후(검증용)** 로 나눕니다. 모델은 앞부분만 보고, 뒷부분으로 채점합니다.

In [ ]:
train = b5[b5.cycle <= 60]
test  = b5[b5.cycle >  60]

print("학습(기출):", len(train), "회  ·  검증(모의고사):", len(test), "회")

### 3. 선형회귀 — fit() 한 줄
이 셀이 하는 일 — `LinearRegression` 모델을 만들고 `fit(입력, 정답)` 으로 **학습**시킵니다. 입력은 사이클, 정답은 용량.
학습이 끝나면 모델 안에 기울기(`coef_`)와 절편(`intercept_`)이 들어 있습니다. 내가 끈 직선과 비교해 보세요.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(train[["cycle"]], train.capacity_Ah)     # 입력(표 모양), 정답

print(f"컴퓨터가 찾은 기울기 a = {model.coef_[0]:.5f}")
print(f"컴퓨터가 찾은 절편   b = {model.intercept_:.4f}")

### 4. 채점 — MAE
이 셀이 하는 일 — `predict()` 로 예측하고, `mean_absolute_error` 로 **기출 점수**와 **모의고사 점수**를 냅니다. 작을수록 좋습니다. 단위는 Ah.

In [ ]:
from sklearn.metrics import mean_absolute_error

pred_train = model.predict(train[["cycle"]])
pred_test  = model.predict(test[["cycle"]])

print(f"학습(기출)  MAE: {mean_absolute_error(train.capacity_Ah, pred_train):.4f} Ah")
print(f"검증(모의)  MAE: {mean_absolute_error(test.capacity_Ah,  pred_test):.4f} Ah")
print(f"직선이 1.6 Ah에 닿는 사이클: {(1.6 - model.intercept_) / model.coef_[0]:.0f}   (실제: 75)")

✅ **체크포인트 ③** — MAE 두 개(기출 · 모의)를 채팅에 올려 주세요.

### 5. 그림으로 — 직선은 앞만 보고 뒤를 못 맞춘다
이 셀이 하는 일 — 학습 구간(진한 점) · 검증 구간(연한 점) · 모델 직선 · 80% 선을 한 그림에.

In [ ]:
xs = pd.DataFrame({"cycle": range(1, 169)})     # 1 ~ 168 사이클 표
plt.figure(figsize=(8, 4))
plt.scatter(train.cycle, train.capacity_Ah, s=12, label="train (cycle 1-60)")
plt.scatter(test.cycle,  test.capacity_Ah,  s=12, color="lightgray", label="test (61-168)")
plt.plot(xs, model.predict(xs), color="orange", linewidth=2, label="linear model")
plt.axhline(1.6, color="red", linestyle="--", label="80 %")
plt.xlabel("cycle"); plt.ylabel("capacity (Ah)"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

**읽기** — 늙는 속도는 갈수록 빨라집니다. 직선은 그걸 모릅니다. **사이클 수 하나만으로는 부족**합니다 → 셀이 늙었다는 **다른 단서**를 모델에 줘야 합니다.

## 블록 3 · 단서를 더 주고 · 본 적 없는 셀을 맞힌다

### 6. 단서(특성) 고르기 · 본 적 없는 셀로 시험 보기
이 셀이 하는 일 — 모델에 줄 단서를 고릅니다. **방전 전에 알 수 있는 것**만: 사이클 수 · 직전 충전 기록 · 내부 저항.
그리고 이번엔 **셀 단위**로 나눕니다 — B0005 · B0006으로 배우고, **한 번도 안 본 B0007**로 시험. 마지막 날 「숨긴 셀」과 같은 조건입니다.

In [ ]:
features = ["cycle", "charge_time_s", "charge_max_temp", "cc_charge_time_s", "re_ohm", "rct_ohm"]

data  = summary.dropna(subset=features)            # 내부 저항이 빈 줄은 뺌
train = data[data.cell_id != "B0007"]               # B0005 + B0006 로 배움
test  = data[data.cell_id == "B0007"]               # B0007 은 처음 봄

print("학습:", len(train), "회 ·  검증(B0007):", len(test), "회")
print("단서:", features)

왜 `discharge_time_s`(방전 시간)는 안 넣나요? 방전 시간은 **방전을 끝내야** 압니다. 그때는 이미 용량을 잰 뒤입니다. 시험 끝나고 답지를 보는 것과 같아서 **반칙(누수)** 입니다. 마지막 날 숨긴 셀 입력에도 이 열은 없습니다.

### 7. 선형회귀 — 단서 6개로
이 셀이 하는 일 — 같은 `LinearRegression` 인데 입력이 열 6개입니다. 본 적 없는 B0007의 MAE와, 예측이 1.6 Ah 아래로 처음 내려가는 사이클을 봅니다.

In [ ]:
lr = LinearRegression().fit(train[features], train.capacity_Ah)
pred_lr = lr.predict(test[features])

mae_lr = mean_absolute_error(test.capacity_Ah, pred_lr)
hit_lr = test.cycle[pred_lr <= 1.6].iloc[0]
print(f"선형회귀  B0007 MAE: {mae_lr:.4f} Ah  ·  80% 예측 사이클: {hit_lr}  (실제 86)")

### 8. 랜덤포레스트 — 질문 트리 수백 개가 투표
이 셀이 하는 일 — 모델 이름만 바꿉니다. `RandomForestRegressor` 는 「내부 저항이 0.06보다 큰가?」 같은 **질문을 가지처럼 이어 붙인 트리**를 300개 만들고, 300개의 답을 평균합니다.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=300, random_state=0).fit(train[features], train.capacity_Ah)
pred_rf = rf.predict(test[features])

mae_rf = mean_absolute_error(test.capacity_Ah, pred_rf)
hit_rf = test.cycle[pred_rf <= 1.6].iloc[0]
print(f"랜덤포레스트 B0007 MAE: {mae_rf:.4f} Ah  ·  80% 예측 사이클: {hit_rf}  (실제 86)")

✅ **체크포인트 ④** — 두 모델의 MAE를 채팅에 올려 주세요.

### 9. 그림으로 — 본 적 없는 셀을 얼마나 맞혔나
이 셀이 하는 일 — B0007의 실제 용량(점)과 두 모델의 예측(선)을 사이클 순서로 겹칩니다.

In [ ]:
plt.figure(figsize=(8, 4))
plt.scatter(test.cycle, test.capacity_Ah, s=14, label="B0007 actual")
plt.plot(test.cycle, pred_lr, color="orange", linewidth=2, label=f"linear (MAE {mae_lr:.3f})")
plt.plot(test.cycle, pred_rf, color="green",  linewidth=2, label=f"random forest (MAE {mae_rf:.3f})")
plt.axhline(1.6, color="red", linestyle="--", label="80 %")
plt.xlabel("cycle"); plt.ylabel("capacity (Ah)"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

### 10. 일부러 과적합 만들기 — 답만 외운 학생
이 셀이 하는 일 — 트리 **한 그루**를 깊이 제한 없이 키워, 블록 2처럼 B0005 앞 60사이클 → 뒷부분을 맞히게 합니다. 기출은 100점, 모의고사는?

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tr5 = b5[b5.cycle <= 60]
te5 = b5[b5.cycle >  60]

tree = DecisionTreeRegressor(max_depth=None).fit(tr5[["cycle"]], tr5.capacity_Ah)   # 깊이 제한 없음
print(f"트리  학습(기출) MAE: {mean_absolute_error(tr5.capacity_Ah, tree.predict(tr5[['cycle']])):.4f}")
print(f"트리  검증(모의) MAE: {mean_absolute_error(te5.capacity_Ah, tree.predict(te5[['cycle']])):.4f}")

plt.figure(figsize=(8, 4))
plt.scatter(tr5.cycle, tr5.capacity_Ah, s=12, label="train")
plt.scatter(te5.cycle, te5.capacity_Ah, s=12, color="lightgray", label="test")
plt.plot(xs, tree.predict(xs), color="red", linewidth=1.5, label="deep tree (memorized)")
plt.plot(xs, model.predict(xs), color="orange", linewidth=2, label="linear")
plt.axhline(1.6, color="red", linestyle="--")
plt.xlabel("cycle"); plt.ylabel("capacity (Ah)"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

**읽기** — 빨간 선은 학습 구간의 **튀는 점까지 전부 외웠습니다**(기출 MAE 0). 그런데 검증 구간에서는 마지막에 본 값만 반복합니다. 외운 학생은 새 문제를 못 풉니다. 모델을 고를 때는 **기출 점수가 아니라 모의고사 점수**를 봅니다.

### 11. 오늘의 결정 문장
이 셀이 하는 일 — 보증 기준 사이클을 정하고, 두 모델의 80% 예측 사이클로 **출하 판정**을 냅니다. 모델이 다르면 판정이 달라질 수 있습니다.

In [ ]:
warranty = 80   # ← 보증 사이클 기준을 바꿔 보기 (60 · 80 · 100)

for name, hit in [("선형회귀", hit_lr), ("랜덤포레스트", hit_rf)]:
    verdict = "출하 가능" if hit >= warranty else "출하 불가"
    print(f"{name}: B0007은 사이클 {hit}에 80% → 보증 {warranty}사이클이면 {verdict}")
print("실제: 사이클 86 →", "출하 가능" if 86 >= warranty else "출하 불가")

## 바꿔 보기 미션 (25분)
셀을 복사하지 말고 `# ←` 자리만 바꿔 실행합니다. 결과는 캡처해서 PAD 게시판 2일차에.

### 미션 1 · 단서를 빼고 더해 비교표 만들기
이 셀이 하는 일 — 단서 묶음 여러 개를 차례로 학습시켜 B0007 MAE를 **표 한 장**으로 모읍니다. 묶음을 하나 더 추가해 보세요(예: `["cycle", "charge_max_temp"]`).

In [ ]:
sets = {
    "사이클만":            ["cycle"],
    "사이클+내부 저항":    ["cycle", "re_ohm", "rct_ohm"],
    "단서 6개 전부":       features,
    # ← 여기에 한 줄 추가: "내 묶음": ["cycle", "charge_max_temp"],
}
rows = []
for name, cols in sets.items():
    m_lr = LinearRegression().fit(train[cols], train.capacity_Ah)
    m_rf = RandomForestRegressor(n_estimators=300, random_state=0).fit(train[cols], train.capacity_Ah)
    rows.append([name, len(cols),
                 round(mean_absolute_error(test.capacity_Ah, m_lr.predict(test[cols])), 4),
                 round(mean_absolute_error(test.capacity_Ah, m_rf.predict(test[cols])), 4)])
pd.DataFrame(rows, columns=["단서 묶음", "열 수", "선형회귀 MAE", "랜덤포레스트 MAE"])

✅ **체크포인트 ⑤** — 비교표를 캡처해 PAD 게시판 2일차에 올려 주세요.

### 미션 2 · 시험 셀 바꾸기
6번 셀의 `"B0007"` 을 `"B0006"` 으로 바꾸면(두 군데), B0006을 처음 보는 셀로 시험합니다. 7 · 8번 셀을 다시 실행해 두 모델의 MAE와 80% 예측 사이클을 적으세요. 실제 B0006은 **63**.

In [ ]:
test_cell = "B0007"   # ← "B0006" 으로

train2 = data[data.cell_id != test_cell]
test2  = data[data.cell_id == test_cell]
for name, m in [("선형회귀", LinearRegression()), ("랜덤포레스트", RandomForestRegressor(n_estimators=300, random_state=0))]:
    m.fit(train2[features], train2.capacity_Ah)
    p = m.predict(test2[features])
    print(f"{name}: {test_cell} MAE {mean_absolute_error(test2.capacity_Ah, p):.4f} · 80% 예측 {test2.cycle[p <= 1.6].iloc[0]}")

### 미션 3 (도전) · 트리 깊이 바꾸기
랜덤포레스트의 `max_depth` 를 1 → 3 → 제한 없음(`None`)으로 바꿔 B0007 MAE를 비교하세요. 질문 수가 적으면 덜 외우지만 덜 맞힙니다.

In [ ]:
for depth in [1]:                              # ← [1, 3, None] 으로
    m = RandomForestRegressor(n_estimators=300, max_depth=depth, random_state=0).fit(train[features], train.capacity_Ah)
    print(f"max_depth={depth}: 학습 MAE {mean_absolute_error(train.capacity_Ah, m.predict(train[features])):.4f} · B0007 MAE {mean_absolute_error(test.capacity_Ah, m.predict(test[features])):.4f}")

## 오늘 배운 것 · 내일
- **학습** = 틀린 만큼 고치기(`fit`). **채점** = MAE. 기출(train)이 아니라 **모의고사(test)** 점수로 모델을 고른다.
- **과적합** = 답만 외운 학생. 기출 0점짜리 오차가 모의고사에서 10배로.
- 사이클 수 하나보다 **늙음과 이어진 단서**(내부 저항)가 세다. 방전 뒤에야 아는 값은 **반칙**.
- 모델이 다르면 **결정이 달라진다** — 그래서 MAE부터 본다.
- **내일** — 2일차 모델이 못 맞춘 셀(B0006)을 깊은 모델(신경망)로. 브라우저에서 층과 뉴런을 마우스로 쌓아 본다.

### 과제
미션 1 비교표 캡처 · 미션 2 결과 · 오늘의 결정 문장 한 줄을 PAD 게시판 2일차에.